I have a folder named
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\Raw Data\
```
Which have many folders with the name in the `YYYY-MM-DD` format
within each folder there is a subfolder called `movie`
And within that `movie` folder there are multiple files in the *.avi format.
Those *.avi file names are all in the format: `YYYY-MM-DD_{Animal_name}_trial_{n}.avi`

Now I want to write a python code, that goes through all the sub-folder in the `Raw Data` folder,
And make a csv file, with columns:
* id (Animal name)
* date (in the YYYYMMDD format)

In [19]:
import os
import pandas as pd

# Path to your Raw Data folder
base_path = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\Raw Data"

records = []

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".avi"):
            # Example filename: 2025-09-18_Mouse1_trial_3.avi
            parts = file.split("_")
            if len(parts) >= 3:
                date = parts[0].replace("-", "")  # YYYYMMDD
                animal_name = parts[1]
                records.append({"id": animal_name, "date": date})
            else:
                print("Attention!")

# Convert to pandas DataFrame
df = pd.DataFrame(records)

# Drop duplicates (one row per animal per date)
df = df.drop_duplicates().sort_values(by=["id", "date"]).reset_index(drop=True)


In [20]:
df

,id,date
0,40,20230406
1,9,20240518
2,A,20240305
3,A,20240306
4,A,20240307
...,...,...
770,T,20241111
771,T,20241112
772,T,20241113
773,T,20241114


In [21]:
import re

base_chr2 = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2"

# --- Now scan ChR2 folder ---
# Regex for session folder: YYYY-MM-DD{type}{number}
session_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})([A-Za-z]+)(\d+)")

new_records = []

# Go two levels down in ChR2
for subfolder in os.listdir(base_chr2):
    subfolder_path = os.path.join(base_chr2, subfolder)
    if os.path.isdir(subfolder_path):
        for session_folder in os.listdir(subfolder_path):
            session_path = os.path.join(subfolder_path, session_folder)
            if os.path.isdir(session_path):
                m = session_pattern.match(session_folder)
                if not m:
                    continue
                date_str, session_type, session_num = m.groups()
                date = date_str.replace("-", "")
                day_in_type = int(session_num)

                # Look for AVI files inside
                for file in os.listdir(session_path):
                    if file.endswith(".avi"):
                        parts = file.split("_")
                        if len(parts) >= 3:
                            animal_name = parts[1]

                            # Check if (id, date) exists in df
                            mask = (df["id"] == animal_name) & (df["date"] == date)
                            if mask.any():
                                # If the type and day_in_type columns values is not nan, check if they match
                                if "type" in df.columns and "day_in_type" in df.columns:
                                    existing_type = df.loc[mask, "type"].values[0]
                                    existing_day = df.loc[mask, "day_in_type"].values[0]
                                    if pd.notna(existing_type) and pd.notna(existing_day):
                                        if existing_type != session_type or existing_day != day_in_type:
                                            print(f"Conflict for {animal_name} on {date}: existing ({existing_type}, {existing_day}) vs new ({session_type}, {day_in_type})")
                                            continue  # Skip this entry
                                else:
                                    df["type"] = pd.NA
                                    df["day_in_type"] = pd.NA
                                # Update df with type and day_in_type
                                df.loc[mask, "type"] = session_type
                                df.loc[mask, "day_in_type"] = day_in_type
                            else:
                                print(f"{animal_name} - {date} does not exist")

In [1]:
import os
import pandas as pd
import re

# Path to your Raw Data folder
base_path = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\Raw Data"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

records = []

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".avi"):
            m = pattern.match(file)
            if m:
                date_str, animal_name, trial_num = m.groups()
                date = date_str.replace("-", "")  # YYYYMMDD
                # if date == '20250607' and animal_name == 'AD':
                #     print(f"Found: {file}")
                trial_num = int(trial_num)
                records.append({
                    "id": animal_name,
                    "date": date,
                    "trial": trial_num
                })
            else:
                print(f"⚠️ Filename format unexpected: {file}")

# Make DataFrame
df_trials = pd.DataFrame(records)

# Group by id/date to compute number of trials
def check_trials(trials, animal_name, date):
    trials_sorted = sorted(trials)
    n_trials = len(trials_sorted)
    max_trial = max(trials_sorted)
    if trials_sorted != list(range(1, max_trial + 1)):
        print(f"⚠️ {animal_name}-{date}: Missing trial(s) in sequence: {trials_sorted}")
    return n_trials

trial_info = (
    df_trials.groupby(["id", "date"])["trial"]
    .apply(list)
    .reset_index()
)

trial_info["n_trials"] = trial_info.apply(
    lambda row: check_trials(row["trial"], row["id"], row["date"]), axis=1
)

# Merge into final df (one row per id/date)
df = trial_info.drop(columns="trial").sort_values(by=["id", "date"]).reset_index(drop=True)

print(df.head())

# # Save CSV
# output_csv = os.path.join(base_path, "animal_data_with_trials.csv")
# df.to_csv(output_csv, index=False)
# print(f"✅ CSV saved to {output_csv}")


⚠️ Filename format unexpected: 20230406_40_OPME230310_P1_pup1F_HET_MOI25000_noAraC_DIV27.czi - C=0.avi
⚠️ Filename format unexpected: 2024-12-11_B_trial_1_REPEAT.avi
⚠️ Filename format unexpected: 2025-06-08_AG_trial_1E.avi
⚠️ 9-20240518: Missing trial(s) in sequence: [10]
⚠️ A-20240305: Missing trial(s) in sequence: [15]
⚠️ AD-20250607: Missing trial(s) in sequence: [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8]
⚠️ AE-20250607: Missing trial(s) in sequence: [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8]
⚠️ AE-20250611: Missing trial(s) in sequence: [1, 3]
⚠️ AF-20250607: Missing trial(s) in sequence: [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8]
⚠️ AF-20250608: Missing trial(s) in sequence: [1, 3, 4]
⚠️ AG-20250607: Missing trial(s) in sequence: [1, 1, 2, 3, 4, 5, 6, 7, 8]
⚠️ AG-20250608: Missing trial(s) in sequence: [2, 3, 4, 5, 6, 7, 8]
⚠️ AH-20250611: Missing trial(s) in sequence: [1, 3, 5]
⚠️ AI-20250716: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12]
⚠️ AJ

Now, with the df created, I want to look into another folder, named:
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2
```
One level down in this folder, there are 6 sub folders, the names of these subfolders does not matter.
But now one more level into each of the 6 subfolders, there are more sub folders, named in the format:
`YYYY-MM-DD{type}{number}`, 
`type` means the session type, `number` is the days since the start of that session type
into these `YYYY-MM-DD{type}{number}` subfolders there are many *.avi files, again they are named
YYYY-MM-DD_{Animal_name}_trial_{n}.avi

I want to write a script, to go through each *.avi files, and
Find the matching animals - date in the `df`.
Add a column `type` in `df` to save the session type information,
And also add a column `day_in_type` to save the `number` information

If that animals - date pair does not exist in `df`, print :
f'{id} - {date} does not exist'

In [2]:
import re

base_chr2 = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2"

# --- Now scan ChR2 folder ---
# Regex for session folder: YYYY-MM-DD{type}{number}
session_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})([A-Za-z]+)(\d+)")

new_records = []

# Go two levels down in ChR2
for subfolder in os.listdir(base_chr2):
    subfolder_path = os.path.join(base_chr2, subfolder)
    if os.path.isdir(subfolder_path):
        for session_folder in os.listdir(subfolder_path):
            session_path = os.path.join(subfolder_path, session_folder)
            if os.path.isdir(session_path):
                m = session_pattern.match(session_folder)
                if not m:
                    continue
                date_str, session_type, session_num = m.groups()
                date = date_str.replace("-", "")
                day_in_type = int(session_num)

                # Look for AVI files inside
                for file in os.listdir(session_path):
                    if file.endswith(".avi"):
                        parts = file.split("_")
                        if len(parts) >= 3:
                            animal_name = parts[1]

                            # Check if (id, date) exists in df
                            mask = (df["id"] == animal_name) & (df["date"] == date)
                            if mask.any():
                                # If the type and day_in_type columns values is not nan, check if they match
                                if "type" in df.columns and "day_in_type" in df.columns:
                                    existing_type = df.loc[mask, "type"].values[0]
                                    existing_day = df.loc[mask, "day_in_type"].values[0]
                                    if pd.notna(existing_type) and pd.notna(existing_day):
                                        if existing_type != session_type or existing_day != day_in_type:
                                            print(f"Conflict for {animal_name} on {date}: existing ({existing_type}, {existing_day}) vs new ({session_type}, {day_in_type})")
                                            continue  # Skip this entry
                                else:
                                    df["type"] = pd.NA
                                    df["day_in_type"] = pd.NA
                                # Update df with type and day_in_type
                                df.loc[mask, "type"] = session_type
                                df.loc[mask, "day_in_type"] = day_in_type
                            else:
                                print(f"{animal_name} - {date} does not exist")

In [3]:
# Save df as csv
df.to_csv("animal_sessions.csv", index=False)